In [5]:
# %%
import time
import numpy as np
import os
from tools import *

from IPython import get_ipython

# Ensure the Qt event loop is integrated with Jupyter
# get_ipython().run_line_magic('gui', 'qt')
from PySide2.QtCore import QTimer, QTime, Signal
%gui qt

In [3]:
measurement_mode(switchlogic, 
                powercontroller_logic, 
                ibeam_smart_remote, 
                mode = 'PLE')

In [4]:
measurement_mode(switchlogic, 
                powercontroller_logic, 
                ibeam_smart_remote, 
                mode = 'Off-res')

In [5]:
# switchlogic.set_state(switch = "Mirror", state = "Off")

In [41]:
switchlogic.set_state(switch = "Shutter", state = "Off")

In [32]:
ibeam_smart_remote.power = 30000


In [31]:
powercontroller_logic._current_motor = 2
powercontroller_logic.motor_position = 190 # MAX green

In [2]:
powercontroller_logic._current_motor = 2
powercontroller_logic.motor_position = 45 # green dim

In [14]:
powercontroller_logic._current_motor = 0
powercontroller_logic.motor_position = 22 # parallel pol

In [28]:
powercontroller_logic._current_motor = 0
powercontroller_logic.motor_position = 2 # perpendicular pol

In [35]:
cts_t = []
def check_counts():
    #%%
    ch1_cts = timetaggerlogic.counter.getDataNormalized()[0, :]
    ch2_cts = timetaggerlogic.counter.getDataNormalized()[1, :]

    # %%
    tot_cts = ch1_cts.mean() + ch2_cts.mean()
    # %%
    # cts_t.append(tot_cts / 1e3)
    # if cts_t <= 6500:

In [42]:
ibeam_smart_remote.power = 10000

In [33]:
timer = QTimer()
timer.timeout.connect(check_counts)
timer.start(1000)  # 1 second interval

In [29]:
cts_t

[8.302200000000001, 8.3115, 8.312700000000001, 8.313799999999999, 8.2871, 8.29, 8.2641, 8.2642, 8.2789, 8.265799999999999, 8.238700000000001, 8.2607, 8.2406, 8.2499]

In [35]:
timer.stop()

In [40]:
ibeam_smart_remote.power = 200
powercontroller_logic._current_motor = 2
powercontroller_logic.motor_position = 210

In [33]:
poi_manager_logic_remote._optimizelogic().start_optimize()

0

In [41]:
measurement_mode(switchlogic, 
                powercontroller_logic, 
                ibeam_smart_remote, 
                mode = 'PLE')

In [37]:
measurement_mode(switchlogic, 
                powercontroller_logic, 
                ibeam_smart_remote, 
                mode = 'Off-res')

In [71]:
powercontroller_logic._current_motor = 0
powercontroller_logic.motor_position = 7 # perpendicular pol

In [59]:
powercontroller_logic._current_motor = 0
powercontroller_logic.motor_position = 30 # perpendicular pol

## Power dependent g2

In [29]:
cts_t = []
def check_counts():
    #%%
    ch1_cts = timetaggerlogic.counter.getDataNormalized()[0, :]
    ch2_cts = timetaggerlogic.counter.getDataNormalized()[1, :]

    # %%
    tot_cts = ch1_cts.mean() + ch2_cts.mean()
    # %%
    
    if tot_cts <= 5000:
        cts_t.append(tot_cts / 1e3)
        powercontroller_logic._current_motor = 0
        powercontroller_logic.motor_position = 7 # perpendicular pol
        time.sleep(3)

        poi_manager_logic_remote._optimizelogic().start_optimize()
        while poi_manager_logic_remote._optimizelogic().module_state()=='locked':
            time.sleep(5)
        powercontroller_logic._current_motor = 0
        powercontroller_logic.motor_position = 30 # parallel pol
    timer.start(200 * 1000)

cts_refocus = []
def refocus():
    #%%
    ch1_cts = timetaggerlogic.counter.getDataNormalized()[0, :]
    ch2_cts = timetaggerlogic.counter.getDataNormalized()[1, :]

    # %%
    tot_cts = ch1_cts.mean() + ch2_cts.mean()
    # %%
    

    cts_refocus.append(tot_cts / 1e3)
    powercontroller_logic._current_motor = 0
    powercontroller_logic.motor_position = 7 # perpendicular pol
    time.sleep(3)

    poi_manager_logic_remote._optimizelogic().start_optimize()
    while poi_manager_logic_remote._optimizelogic().module_state()=='locked':
        time.sleep(1) # wait for a long time to 
    time.sleep(20) # wait for a long time to avoid conflicts with the countrate checker

    powercontroller_logic._current_motor = 0
    powercontroller_logic.motor_position = 30 # parallel pol
    time.sleep(3)

In [53]:
poi_manager_logic_remote._optimizelogic()._last_fit_results.rsquared

0.8481955553905736

## Power dependent g2

In [30]:
folder_g2_save = r'Z:\Vlad\SnV\TPI\Electrodes_e4\F1\atto3_D1\test_auto1'
current_cryo = 'atto3'
def g2_power_dependent():
    if len(powers) < 1:
        integration_timer.stop()
        refocus_timer.stop()
        timer.stop()
    power = powers.pop()
    if current_cryo == 'atto3':
        ibeam_smart_remote.power = power
    else:
        powercontroller_logic._current_motor = 2
        powercontroller_logic.motor_position = power # perpendicular pol

    timetagger._mw.dump_checkBox.setChecked(False)
    timetagger._dump_toggled()

    measurement_mode(switchlogic, 
                powercontroller_logic, 
                ibeam_smart_remote, 
                mode = 'Off-res')
    
    powercontroller_logic._current_motor = 0
    powercontroller_logic.motor_position = 7 # perpendicular pol
    refocus()
    powercontroller_logic._current_motor = 0
    powercontroller_logic.motor_position = 30 # parallel pol

    # start the measurement:
    
    pth = os.path.join(folder_g2_save, str(power))
    os.makedirs(pth, exist_ok = True)
    timetagger._mw.saveDumpTagLineEdit.setText(power)
    timetagger._save_dump_folderpath = pth
    timetagger._mw.currDumpPathLabel.setText(timetagger._save_dump_folderpath)
    
    timetagger._mw.dump_checkBox.setChecked(True)
    timetagger._dump_toggled()

In [31]:
timer = QTimer()
timer.timeout.connect(check_counts) # rough check on drift to ensure it doesnt fly away
timer.setSingleShot(True)
timer.start(10000)  # 1 second interval

refocus_timer = QTimer()
refocus_timer.timeout.connect(check_counts)
# refocus_timer.setSingleShot(True)
refocus_timer.start(25 * 60000)  # each 20 mins second interval

integration_timer = QTimer()
integration_timer.timeout.connect(g2) # rough check on drift to ensure it doesnt fly away
integration_timer.setSingleShot(True)



In [33]:
atto3_powers = [5e3, 10e3, 15e3, 20e3, 30e3, 40e3]
bf_powers = [100, 125, 150, 175] #tentative

current_cryo = 'atto3'
powers = atto3_powers
# powers = bf_powers

integrate_for_mins = 120
integration_timer.start(integrate_for_mins * 60e3)  # integrate in minutes

In [34]:
refocus_timer.stop()
timer.stop()
integration_timer.stop()

## Voltage dependent g2 HOM

In [46]:
ao_electrodes_remote.sig_set_setpoint = Signal()

TypeError: cannot pickle 'PySide2.QtCore.Signal' object

========= Remote Traceback (2) =========
Traceback (most recent call last):
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 369, in _dispatch_request
    res = self._HANDLERS[handler](self, *args)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 885, in _handle_setattr
    return self._access_attr(obj, name, (value,), "_rpyc_setattr", "allow_setattr", setattr)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 807, in _access_attr
    return accessor(obj, name, *args)
  File "C:\Users\YY3\GIT\squdi-core\src\qudi\core\services.py", line 261, in __setattr__
    return setattr(obj, name, netobtain(value))
  File "C:\Users\YY3\GIT\squdi-core\src\qudi\util\network.py", line 32, in netobtain
    return _classic.obtain(obj)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\utils\classic.py", line 296, in 

In [43]:
ao_electrodes_remote._create_ao_task

TypeError: __annotations__ must be set to a dict object

========= Remote Traceback (1) =========
Traceback (most recent call last):
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 369, in _dispatch_request
    res = self._HANDLERS[handler](self, *args)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 879, in _handle_getattr
    return self._access_attr(obj, name, (), "_rpyc_getattr", "allow_getattr", getattr)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 807, in _access_attr
    return accessor(obj, name, *args)
  File "C:\Users\YY3\GIT\squdi-core\src\qudi\core\services.py", line 245, in __getattribute__
    def wrapped(*args, **kwargs):
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\functools.py", line 56, in update_wrapper
    setattr(wrapper, attr, value)
TypeError: __annotations__ must be set to a dict object


In [41]:
ao_electrodes_remote.set_setpoint(channel='ao3', value=0)

TypeError: __annotations__ must be set to a dict object

========= Remote Traceback (1) =========
Traceback (most recent call last):
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 369, in _dispatch_request
    res = self._HANDLERS[handler](self, *args)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 879, in _handle_getattr
    return self._access_attr(obj, name, (), "_rpyc_getattr", "allow_getattr", getattr)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 807, in _access_attr
    return accessor(obj, name, *args)
  File "C:\Users\YY3\GIT\squdi-core\src\qudi\core\services.py", line 245, in __getattribute__
    def wrapped(*args, **kwargs):
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\functools.py", line 56, in update_wrapper
    setattr(wrapper, attr, value)
TypeError: __annotations__ must be set to a dict object


In [35]:
ao_electrodes_remote.set_activity_state("ao3", True)
ao_electrodes_remote.set_setpoint({"ao3": 0})

TypeError: __annotations__ must be set to a dict object

========= Remote Traceback (1) =========
Traceback (most recent call last):
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 369, in _dispatch_request
    res = self._HANDLERS[handler](self, *args)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 879, in _handle_getattr
    return self._access_attr(obj, name, (), "_rpyc_getattr", "allow_getattr", getattr)
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 807, in _access_attr
    return accessor(obj, name, *args)
  File "C:\Users\YY3\GIT\squdi-core\src\qudi\core\services.py", line 245, in __getattribute__
    def wrapped(*args, **kwargs):
  File "C:\Users\YY3\anaconda3\envs\squdi\lib\functools.py", line 56, in update_wrapper
    setattr(wrapper, attr, value)
TypeError: __annotations__ must be set to a dict object


In [ ]:
folder_g2_save = r'Z:\Vlad\SnV\TPI\Electrodes_e4\F1\atto3_D1\test_auto1'
current_cryo = 'atto3'
def V_dependent_hom():
    if len(voltages) < 1:
        integration_timer.stop()
        refocus_timer.stop()
        timer.stop()
    voltage = voltages.pop()
    
    ao_electrodes_remote.set_activity_state("ao3", True)
    ao_electrodes_remote.set_setpoint({"ao3": voltage})

    timetagger._mw.dump_checkBox.setChecked(False)
    timetagger._dump_toggled()

    measurement_mode(switchlogic, 
                powercontroller_logic, 
                ibeam_smart_remote, 
                mode = 'Off-res')
    
    powercontroller_logic._current_motor = 0
    powercontroller_logic.motor_position = 7 # perpendicular pol
    refocus()
    powercontroller_logic._current_motor = 0
    powercontroller_logic.motor_position = 30 # parallel pol

    # start the measurement:
    
    pth = os.path.join(folder_g2_save, str(power))
    os.makedirs(pth, exist_ok = True)
    timetagger._mw.saveDumpTagLineEdit.setText(power)
    timetagger._save_dump_folderpath = pth
    timetagger._mw.currDumpPathLabel.setText(timetagger._save_dump_folderpath)
    
    timetagger._mw.dump_checkBox.setChecked(True)
    timetagger._dump_toggled()